# 2 — Extractor TensorRT export

**Prerequisites:**
- `uv sync --group trt` (or `--all-groups`) for `polygraphy`, `tensorrt`
- `pip install onnxsim` (or add `onnxsim` to the `trt` dependency group in `pyproject.toml`)
- A CUDA-capable GPU with TensorRT installed

**Prerequisite:** Run [1-export_extractors.ipynb](1-export_extractors.ipynb) first to generate the `.onnx` files in `weights/euroc/`.

In [ ]:
from pathlib import Path

CWD = Path.cwd().resolve()
if CWD.name == "notebooks":
    LG = CWD.parent
elif CWD.name == "LightGlue-ONNX-Jetson":
    LG = CWD
else:
    raise SystemExit("Run Jupyter with cwd = LightGlue-ONNX-Jetson or LightGlue-ONNX-Jetson/notebooks")

OUT_DIR = LG / "weights" / "euroc"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("LG     ", LG)
print("OUT_DIR", OUT_DIR, OUT_DIR.exists())

### ONNX models to process

In [ ]:
# List only the original (non-simplified) ONNX models
onnx_models = sorted(p for p in OUT_DIR.glob("*.onnx") if "_sim" not in p.stem)
for p in onnx_models:
    print(p.name, f"  {p.stat().st_size / 1e6:.1f} MB")

if not onnx_models:
    raise SystemExit("No ONNX models found in OUT_DIR. Run export_extractors.ipynb first.")

### ONNX simplification (onnxsim)

Runs `onnxsim` on each model. Constant folding and dead-node elimination reduce graph size
and improve TensorRT compilation speed. Output: `*_sim.onnx` alongside the originals.

In [ ]:
import sys

if str(LG) not in sys.path:
    sys.path.insert(0, str(LG))

from lightglue_dynamo.extractor_export import simplify_onnx

sim_paths: dict[str, Path] = {}
for onnx_path in onnx_models:
    sim_path = onnx_path.with_name(onnx_path.stem + "_sim.onnx")
    try:
        simplify_onnx(onnx_path, sim_path)
        ratio = sim_path.stat().st_size / onnx_path.stat().st_size
        # Growth >10% is typically DeformConv Constant materialisation — expected and correct.
        flag = " [grew — Constant→initializer materialisation, check=True]" if ratio > 1.1 else ""
        print(f"[OK]   {onnx_path.name}")
        print(f"       -> {sim_path.name}  ({ratio:.1%} of original){flag}")
        sim_paths[onnx_path.stem] = sim_path
    except Exception as exc:
        print(f"[ERR]  {onnx_path.name}: {exc} — using original")
        sim_paths[onnx_path.stem] = onnx_path

print(f"\nSimplified {sum(1 for p in sim_paths.values() if '_sim' in p.stem)}/{len(onnx_models)} models.")

### TensorRT engine build (FP16)

Builds one FP16 engine per simplified ONNX. Engines are saved as `*_sim.fp16.engine`.

> **ALIKED note:** ALIKED uses `DeformConv` (ONNX opset 19).

In [ ]:
from lightglue_dynamo.extractor_export import build_extractor_trt_engine

engine_paths: dict[str, Path] = {}
for stem, sim_path in sim_paths.items():
    engine_path = sim_path.with_suffix(".fp16.engine")
    try:
        build_extractor_trt_engine(sim_path, engine_path, fp16=True)
        print(f"[OK]   {engine_path.name}  ({engine_path.stat().st_size / 1e6:.1f} MB)")
        engine_paths[stem] = engine_path
    except Exception as exc:
        print(f"[FAIL] {stem}: {exc}")

print(f"\nBuilt {len(engine_paths)}/{len(sim_paths)} TRT engines.")

### Test — ORT CUDA vs TRT

Runs both ORT CUDA (on the simplified ONNX) and TRT (on the engine) with the same
test images. Reports output shapes and max absolute difference for float tensors.

In [ ]:
from lightglue_dynamo.extractor_export import EXTRACTOR_REGISTRY


def parse_engine_stem(stem: str) -> tuple[str | None, str]:
    """Parse engine stem -> (extractor_id, variant).

    Variant is one of 'monolithic', 'dense', 'dhead'. ALIKED dense-split
    engines are detected by the '_dense_' / '_dhead_' marker; everything
    else falls back to longest-prefix match against the registry.
    """
    for marker, variant in (("_dense_", "dense"), ("_dhead_", "dhead")):
        if marker in stem:
            return stem.split(marker)[0], variant
    for eid in sorted(EXTRACTOR_REGISTRY, key=len, reverse=True):
        if stem.startswith(eid):
            return eid, "monolithic"
    return None, "monolithic"

In [ ]:
import re
import time

import cv2
import numpy as np
import onnxruntime as ort
import torch
from polygraphy.backend.common import BytesFromPath
from polygraphy.backend.trt import EngineFromBytes, TrtRunner

PROVIDERS_ORT = ["CUDAExecutionProvider", "CPUExecutionProvider"]
N_WARMUP = 5
N_TIMED = 20
asset_dir = LG / "assets"
img_files = ["sacre_coeur1.jpg", "sacre_coeur2.jpg"]
ALIKED_WEIGHTS_DIR = LG.parent / "weights"  # REPO_ROOT/weights/aliked-*.pth

_ALIKED_PT_NAME = {
    "aliked_n16": "aliked-n16",
    "aliked_n16rot": "aliked-n16rot",
    "aliked_n32": "aliked-n32",
    "aliked_t16": "aliked-t16",
}
_aliked_pt_cache: dict[str, torch.nn.Module] = {}


def parse_hw_from_stem(stem: str) -> tuple[int, int]:
    m = re.search(r"_h(\d+)_w(\d+)", stem)
    if m is None:
        raise ValueError(f"Cannot parse H/W from model stem: {stem}")
    return int(m.group(1)), int(m.group(2))


def parse_b_from_stem(stem: str) -> int:
    m = re.search(r"_b(\d+)_", stem)
    return int(m.group(1)) if m else 1


def parse_kp_from_stem(stem: str) -> int:
    m = re.search(r"_kp(\d+)", stem)
    return int(m.group(1)) if m else 256


def load_imgs_bgr(h: int, w: int, n: int) -> list[np.ndarray]:
    imgs = []
    for f in img_files:
        p = asset_dir / f
        if p.is_file():
            imgs.append(cv2.resize(cv2.imread(str(p)), (w, h)))
        else:
            print(f"[WARN] {f} not found - using random noise image")
            imgs.append(np.random.randint(0, 255, (h, w, 3), dtype=np.uint8))
    # Repeat to satisfy the engine's batch dim if we have fewer assets than B.
    while len(imgs) < n:
        imgs.append(imgs[len(imgs) % 2])
    return imgs[:n]


def make_batch(spec, imgs):
    frames = []
    for img in imgs:
        p = spec.preprocess(img)
        while p.ndim > 3:
            p = p[0]
        frames.append(p)
    return np.stack(frames, axis=0)


def get_pytorch_aliked(eid: str) -> torch.nn.Module:
    """Lazy-load PyTorch ALIKED reference (CPU fp32) for diffing TRT outputs."""
    if eid in _aliked_pt_cache:
        return _aliked_pt_cache[eid]
    if eid not in _ALIKED_PT_NAME:
        raise KeyError(f"No PyTorch ALIKED mapping for {eid!r}")
    from lightglue_dynamo.models import ALIKED

    name = _ALIKED_PT_NAME[eid]
    ck = ALIKED_WEIGHTS_DIR / f"{name}.pth"
    if not ck.is_file():
        raise FileNotFoundError(f"PyTorch ALIKED checkpoint not found: {ck}")
    m = ALIKED(model_name=name, device="cpu", pretrained_path=str(ck)).eval()
    _aliked_pt_cache[eid] = m
    return m


def run_trt(engine_path, feed_dict):
    """Warmup + timed TRT run. Returns (outputs_copied_to_host, ms_per_batch)."""
    with TrtRunner(EngineFromBytes(BytesFromPath(str(engine_path)))) as runner:
        for _ in range(N_WARMUP):
            outs = runner.infer(feed_dict=feed_dict)
        t0 = time.perf_counter()
        for _ in range(N_TIMED):
            outs = runner.infer(feed_dict=feed_dict)
        elapsed_ms = (time.perf_counter() - t0) / N_TIMED * 1000
        outs_copy = {k: np.asarray(v).copy() for k, v in outs.items()}
    return outs_copy, elapsed_ms


# Allow this cell to run standalone (skip cells 3-7) by discovering existing
# engines / sim ONNX files from disk when those dicts haven't been built.
try:
    engine_paths  # noqa: F821
except NameError:
    engine_paths = {p.stem.removesuffix(".fp16"): p for p in sorted(OUT_DIR.glob("*.fp16.engine"))}
    print(f"[info] engine_paths discovered from disk: {len(engine_paths)} engine(s)")
try:
    sim_paths  # noqa: F821
except NameError:
    sim_paths = {p.stem: p for p in sorted(OUT_DIR.glob("*_sim.onnx"))}


for stem, engine_path in engine_paths.items():
    eid, variant = parse_engine_stem(stem)
    if eid is None:
        print(f"\n[SKIP] {stem}: cannot resolve extractor id")
        continue

    h, w = parse_hw_from_stem(stem)
    b = parse_b_from_stem(stem)
    imgs_bgr = load_imgs_bgr(h, w, b)
    is_aliked = eid.startswith("aliked")

    print(f"\n=== {eid} / {variant} (B={b}, H={h}, W={w}) ===")

    # ----- ALIKED dense: PyTorch ref via extract_dense_map -----
    if is_aliked and variant == "dense":
        spec = EXTRACTOR_REGISTRY[eid]
        x = make_batch(spec, imgs_bgr)
        trt_outs, ms = run_trt(engine_path, {"image": x})
        print(f"  TRT {ms:.2f} ms/batch  ({ms / b:.2f} ms/img)  (ORT skipped: DeformConv unsupported on CUDA EP)")
        try:
            m = get_pytorch_aliked(eid)
            with torch.inference_mode():
                feat_pt, score_pt = m.extract_dense_map(torch.from_numpy(x))
            for name, ref_t in (("score_map", score_pt), ("feature_map", feat_pt)):
                trt_t = trt_outs.get(name)
                if trt_t is None:
                    print(f"  {name}: TRT output missing")
                    continue
                d = float(np.abs(ref_t.numpy().astype(np.float32) - trt_t.astype(np.float32)).max())
                print(f"  {name}: shape={trt_t.shape} dtype={trt_t.dtype} max_diff_vs_pytorch={d:.5f}")
        except Exception as exc:
            print(f"  [WARN] PyTorch ref unavailable: {exc}")
            for name, t in trt_outs.items():
                print(f"  {name}: shape={t.shape} dtype={t.dtype}")
        continue

    # ----- ALIKED dhead: PyTorch ref via desc_head.forward_batched_fixed_k -----
    if is_aliked and variant == "dhead":
        spec = EXTRACTOR_REGISTRY[eid]
        x = make_batch(spec, imgs_bgr)
        kp = parse_kp_from_stem(stem)
        try:
            m = get_pytorch_aliked(eid)
            with torch.inference_mode():
                feat_pt, _ = m.extract_dense_map(torch.from_numpy(x))
            rng = np.random.default_rng(0)
            kpts_n = rng.uniform(-0.9, 0.9, size=(b, kp, 2)).astype(np.float32)

            feat_np = feat_pt.numpy().astype(np.float32)
            trt_outs, ms = run_trt(engine_path, {"feature_map": feat_np, "kpts_n": kpts_n})
            print(f"  TRT {ms:.2f} ms/batch  ({ms / b:.2f} ms/img)  (ORT skipped)")

            with torch.inference_mode():
                desc_pt = m.desc_head.forward_batched_fixed_k(feat_pt, torch.from_numpy(kpts_n))
            trt_d = trt_outs["descriptors"]
            d = float(np.abs(desc_pt.numpy().astype(np.float32) - trt_d.astype(np.float32)).max())
            print(f"  descriptors: shape={trt_d.shape} dtype={trt_d.dtype} max_diff_vs_pytorch={d:.5f}")
        except Exception as exc:
            print(f"  [WARN] PyTorch ref unavailable: {exc}")
        continue

    # ----- ALIKED monolithic: TRT only (skip ORT - DeformConv unsupported on CUDA EP) -----
    if is_aliked:
        spec = EXTRACTOR_REGISTRY[eid]
        x = make_batch(spec, imgs_bgr)
        trt_outs, ms = run_trt(engine_path, {spec.input_name: x})
        print(f"  TRT {ms:.2f} ms/batch  ({ms / b:.2f} ms/img)  (ORT skipped: DeformConv unsupported on CUDA EP)")
        for name in spec.output_names:
            t = trt_outs.get(name)
            if t is None:
                print(f"  {name}: TRT output missing")
            else:
                print(f"  {name}: shape={t.shape} dtype={t.dtype}")
        continue

    # ----- Non-ALIKED extractors: ORT vs TRT diff path -----
    spec = EXTRACTOR_REGISTRY[eid]
    x = make_batch(spec, imgs_bgr)
    sim_path = sim_paths.get(stem)

    ort_outs = None
    ort_ep = None
    last_ort_exc = "no _sim.onnx alongside engine" if sim_path is None else None
    if sim_path is not None:
        for providers in [PROVIDERS_ORT, ["CPUExecutionProvider"]]:
            try:
                sess = ort.InferenceSession(str(sim_path), providers=providers)
                ort_outs = sess.run(None, {spec.input_name: x})
                ort_ep = providers[0].replace("ExecutionProvider", "")
                break
            except Exception as exc:
                last_ort_exc = exc

    trt_outs, ms = run_trt(engine_path, {spec.input_name: x})
    label = "SKIP" if ort_outs is None else ort_ep
    print(f"  ORT ref: {label}  TRT {ms:.2f} ms/batch  ({ms / b:.2f} ms/img)")

    if ort_outs is None:
        print(f"  skip reason: {last_ort_exc}")
        for name, t in trt_outs.items():
            print(f"  {name}: shape={t.shape} dtype={t.dtype}")
    else:
        for i, name in enumerate(spec.output_names):
            ort_t = ort_outs[i]
            trt_t = trt_outs.get(name)
            if trt_t is None:
                print(f"  {name}: TRT output missing")
                continue
            if np.issubdtype(ort_t.dtype, np.floating):
                d = float(np.abs(ort_t.astype(np.float32) - trt_t.astype(np.float32)).max())
                print(f"  {name}: shape={ort_t.shape} dtype={ort_t.dtype} max_diff={d:.5f}")
            else:
                match = np.array_equal(ort_t, trt_t)
                print(f"  {name}: shape={ort_t.shape} dtype={ort_t.dtype} exact_match={match}")

### Keypoint visualisation (TRT outputs)

In [ ]:
import matplotlib.pyplot as plt

# Discover engines directly from disk so the cell works without re-running the build.
all_engines = sorted(OUT_DIR.glob("*.fp16.engine"))


def _stem_no_fp16(p):
    return p.with_suffix("").stem.removesuffix(".fp16")


# Visualise both monolithic engines (kpts come from the engine) and dense engines
# (kpts derived from score_map via PyTorch DKD postproc). Skip dhead (no kpts).
viz_engines: dict[str, tuple[Path, str]] = {}
for p in all_engines:
    stem = _stem_no_fp16(p)
    eid, variant = parse_engine_stem(stem)
    if eid is None or variant == "dhead":
        continue
    viz_engines[stem] = (p, variant)

print(f"Found {len(all_engines)} engine(s); visualising {len(viz_engines)}.")
for stem, (_, variant) in viz_engines.items():
    print(f"  [{variant:10}] {stem}")

DKD_TOPK = 256  # how many kpts to plot for dense engines


def dense_kpts_from_score_map(eid: str, score_map_np: np.ndarray, top_k: int):
    """Run PyTorch DKD postproc on a TRT score_map -> (kpts_norm[B,K,2], scores[B,K])."""
    m = get_pytorch_aliked(eid)
    # configure_aliked_for_export normally bumps these; mirror it for the viz path.
    m.dkd.top_k = top_k
    m.dkd.n_limit = top_k
    score_t = torch.from_numpy(score_map_np.astype(np.float32))
    with torch.inference_mode():
        kpts_list, _, scores_list = m.dkd.detect_keypoints(score_t, sub_pixel=True)
    kpts_b = torch.stack(kpts_list).numpy()
    scores_b = torch.stack(scores_list).numpy()
    return kpts_b, scores_b


if not viz_engines:
    print("No monolithic / dense engines to visualise.")
else:
    n_rows = len(viz_engines)
    n_cols = len(img_files)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows), dpi=96)
    if n_rows == 1:
        axes = np.expand_dims(axes, 0)
    fig.suptitle("TRT Extractor Smoke Test  (per-model H/W from engine name, FP16)", fontsize=13)

    def to_pixel_coords(kpts: np.ndarray, h: int, w: int) -> np.ndarray:
        if np.all((kpts >= -1.1) & (kpts <= 1.1)):
            wh = np.array([w - 1, h - 1], dtype=np.float32)
            return (kpts + 1) / 2 * wh
        return kpts

    for row_i, (stem, (engine_path, variant)) in enumerate(viz_engines.items()):
        eid, _ = parse_engine_stem(stem)
        spec = EXTRACTOR_REGISTRY[eid]

        h, w = parse_hw_from_stem(stem)
        b = parse_b_from_stem(stem)
        imgs_bgr = load_imgs_bgr(h, w, b)
        x = make_batch(spec, imgs_bgr)

        if variant == "dense":
            input_name = "image"
        else:
            input_name = spec.input_name

        with TrtRunner(EngineFromBytes(BytesFromPath(str(engine_path)))) as runner:
            trt_outs = runner.infer(feed_dict={input_name: x})
            trt_outs = {k: np.asarray(v).copy() for k, v in trt_outs.items()}

        if variant == "dense":
            # DKD postproc on the dense engine's score_map.
            score_map = trt_outs["score_map"]  # (B, 1, H, W)
            kpts_b, scores_b = dense_kpts_from_score_map(eid, score_map, top_k=DKD_TOPK)
            num_kpts = None
        else:
            kpts_b = trt_outs["keypoints"]
            scores_b = trt_outs["keypoint_scores"]
            num_kpts = trt_outs.get("num_keypoints")

        n_show = min(b, n_cols)
        num_valid = [
            int(num_kpts[i]) if num_kpts is not None else int((scores_b[i] > 0).sum())
            for i in range(n_show)
        ]

        for col_i in range(n_cols):
            ax = axes[row_i, col_i]
            if col_i >= n_show:
                ax.axis("off")
                continue
            img_bgr = imgs_bgr[col_i]
            n = num_valid[col_i]
            h_img, w_img = img_bgr.shape[:2]
            ax.imshow(img_bgr[..., ::-1])
            if n > 0:
                kpts = to_pixel_coords(kpts_b[col_i, :n], h_img, w_img)
                scores = scores_b[col_i, :n]
                sc = ax.scatter(kpts[:, 0], kpts[:, 1], c=scores, cmap="plasma", s=4, linewidths=2,
                                vmin=scores.min(), vmax=scores.max())
                plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.02)
            ax.set_title(f"{eid}/{variant} (H={h},W={w},B={b})  |  {img_files[col_i]}  ({n} kpts)", fontsize=8)
            ax.axis("off")

    plt.tight_layout()
    plt.show()